# Validación cruzada estratificada 5-fold — MLP V1.0 vs V2.0

Este notebook corresponde al **EXP-005** del registro maestro.

## Objetivo

Evaluar la **estabilidad** de las configuraciones V1.0 y V2.0 mediante 5 folds estratificados, evitando usar el conjunto TEST para tomar nuevas decisiones.

## Regla central

El conjunto original se reconstruye exactamente con la misma semilla usada en el preprocesamiento:

- TRAIN original: 699 casos
- VALIDATION original: 150 casos
- TEST original: 150 casos

Para este experimento:

- **Desarrollo = TRAIN + VALIDATION originales = 849 casos**
- **TEST = 150 casos congelados y NO utilizado en cross-validation**

Dentro de cada fold externo:

1. Se separa un fold de evaluación.
2. La porción restante se divide internamente en:
   - entrenamiento efectivo;
   - validación interna para EarlyStopping.
3. El preprocesamiento se ajusta **únicamente con el entrenamiento efectivo**.
4. Se aplican las mismas transformaciones a validación interna y fold de evaluación.
5. V1 y V2 usan exactamente las mismas particiones y matrices transformadas.

## Métrica principal

**PR-AUC** (área trapezoidal bajo la curva Precision-Recall).

También se reportan ROC-AUC, Average Precision, Recall, Precision, F1, MCC, Balanced Accuracy, FNR y matriz de confusión a threshold 0.50.

> Las métricas a threshold 0.50 son secundarias. La comparación de arquitectura se apoya principalmente en métricas independientes del threshold.


## 1) Librerías y configuración


In [ ]:
import os
import re
import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    balanced_accuracy_score,
    matthews_corrcoef
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
N_SPLITS = 5
INNER_VAL_SIZE = 0.15
MAX_EPOCHS = 200
THRESHOLD_DESCRIPTIVE = 0.50

TARGET = "PERMANENCIA_NORMALIZADA"

# Ruta del dataset base usada en el notebook V1 presentado al tutor.
DATASET_PATH = Path("data/Datos_depurados_base.xlsx")

OUTPUT_DIR = Path("outputs/resultados_cv5_v1_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("Determinismo de TensorFlow activado.")
except Exception as e:
    print("No fue posible activar determinismo completo:", e)

# No se requiere montar Google Drive en esta versión pública.


## 2) Carga y validación del dataset crudo

Se parte del dataset **antes de escalado y One-Hot Encoding**. Esto es necesario para que cada fold aprenda sus propios parámetros de preprocesamiento únicamente a partir de sus datos de entrenamiento.


In [ ]:
dataset_path = Path(DATASET_PATH)

if not dataset_path.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {dataset_path}\n"
        "Ajusta DATASET_PATH antes de continuar."
    )

if dataset_path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(dataset_path)
elif dataset_path.suffix.lower() == ".csv":
    df = pd.read_csv(dataset_path, encoding="utf-8-sig")
else:
    raise ValueError("Formato no soportado. Use CSV/XLSX/XLS.")

print("Dimensiones dataset crudo:", df.shape)

if TARGET not in df.columns:
    raise KeyError(f"No se encontró la variable objetivo: {TARGET}")

df_work = df.copy()
df_work["y"] = pd.to_numeric(
    df_work[TARGET],
    errors="coerce"
).astype("Int64")

if df_work["y"].isna().any():
    raise ValueError("El target contiene valores no convertibles a 0/1.")

target_values = sorted(df_work["y"].unique().tolist())

if target_values != [0, 1]:
    raise ValueError(
        f"Se esperaba target binario [0,1], se encontró: {target_values}"
    )

df_work["y"] = df_work["y"].astype(int)

print("\nDistribución objetivo:")
display(
    pd.DataFrame({
        "n": df_work["y"].value_counts().sort_index(),
        "proporción": df_work["y"].value_counts(
            normalize=True
        ).sort_index()
    })
)


## 3) Reconstrucción del split original y congelación de TEST

Se reproduce exactamente el procedimiento del notebook de preprocesamiento original con `random_state=42`.

**Importante:** a partir de aquí TEST se mantiene separado y no se utiliza para entrenar, detener, seleccionar hiperparámetros ni calcular métricas de cross-validation.


In [ ]:
RANDOM_STATE = 42

y_all = df_work["y"].to_numpy()

train_idx, temp_idx = train_test_split(
    df_work.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_all
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df_work.loc[temp_idx, "y"]
)

sizes = (
    len(train_idx),
    len(val_idx),
    len(test_idx)
)

print("TRAIN original:", sizes[0])
print("VAL original  :", sizes[1])
print("TEST congelado:", sizes[2])

EXPECTED = (699, 150, 150)

if sizes != EXPECTED:
    raise ValueError(
        f"El split reproducido {sizes} no coincide con el esperado {EXPECTED}. "
        "Verifica que DATASET_PATH corresponda exactamente al dataset utilizado "
        "para V1/V2."
    )

development_idx = np.concatenate([
    np.asarray(train_idx),
    np.asarray(val_idx)
])

frozen_test_idx = np.asarray(test_idx)

if len(set(development_idx).intersection(set(frozen_test_idx))) != 0:
    raise RuntimeError("Hay solapamiento entre desarrollo y TEST.")

print("\nDesarrollo para CV:", len(development_idx))
print("TEST congelado     :", len(frozen_test_idx))

print("\nDistribución en desarrollo:")
print(
    df_work.loc[development_idx, "y"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nDistribución en TEST congelado (solo descriptiva):")
print(
    df_work.loc[frozen_test_idx, "y"]
    .value_counts(normalize=True)
    .sort_index()
)


## 4) Limpieza determinista

Estas operaciones no aprenden medias, escalas, cuantiles ni categorías. Por eso pueden aplicarse antes del 5-fold sin introducir información aprendida desde los folds de evaluación.


In [ ]:
def normalize_object_series(series):
    return series.apply(
        lambda x: np.nan
        if pd.isna(x)
        else re.sub(r"\s+", " ", str(x).strip())
    )

df_clean = df_work.copy()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = normalize_object_series(df_clean[col])

for col in df_clean.columns:
    if isinstance(col, str) and "%" in col:
        df_clean[col] = pd.to_numeric(
            df_clean[col],
            errors="coerce"
        )

X_raw = df_clean.drop(columns=["y"])

if TARGET in X_raw.columns:
    X_raw = X_raw.drop(columns=[TARGET])

y = df_clean["y"].astype(int).to_numpy()

X_dev_raw = X_raw.loc[development_idx].reset_index(drop=True)
y_dev = df_clean.loc[development_idx, "y"].astype(int).to_numpy()

# Se conserva solo para comprobar que permanece fuera del experimento.
X_test_frozen_raw = X_raw.loc[frozen_test_idx].reset_index(drop=True)
y_test_frozen = df_clean.loc[frozen_test_idx, "y"].astype(int).to_numpy()

print("X desarrollo:", X_dev_raw.shape)
print("y desarrollo:", y_dev.shape)
print("TEST congelado:", X_test_frozen_raw.shape)


## 5) Reglas de preprocesamiento originales


In [ ]:
COUNT_COLS = [
    "Cursos matriculados",
    "Cursos aprobados",
    "Cursos retirados",
    "Cursos reprobados",
    "Cursos desertados",
    "Créditos matriculados sin retiro",
    "Créditos aprobados",
    "Créditos retirados",
    "Créditos reprobados",
    "Créditos desertados",
    "Cursos repetidos",
    "Cantidad de carreras"
]

PERCENT_COLS = [
    "% cursos aprobados",
    "% cursos reprobados",
    "% cursos desertados",
    "% créditos aprobados",
    "% créditos reprobados",
    "% créditos desertados"
]

FORCE_NUMERIC = ["Edad"]
FORCE_CATEGORICAL = ["Código Índice de Urbanidad"]


class ClipTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q=0.0, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.upper_ = np.nanquantile(
            X,
            self.upper_q,
            axis=0
        )
        self.lower_ = (
            np.nanquantile(
                X,
                self.lower_q,
                axis=0
            )
            if self.lower_q > 0
            else None
        )
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        if self.lower_ is not None:
            X = np.maximum(X, self.lower_)

        return np.minimum(X, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.array(
            input_features
            if input_features is not None
            else []
        )


def safe_log1p(x):
    return np.log1p(
        np.maximum(x, 0)
    )


def to_proportion(x):
    return x / 100.0


def astype_str(x):
    return x.astype(str)


def make_onehot():
    # Compatible con versiones nuevas y anteriores de scikit-learn.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )


## 6) Construcción del preprocesador por fold

La selección de variables de baja cardinalidad, medianas, percentil 99, medias/desviaciones y categorías del One-Hot se aprende **solo desde el entrenamiento efectivo de cada fold**.


In [ ]:
def build_preprocessor(X_fit_raw):
    count_cols_present = [
        c for c in COUNT_COLS
        if c in X_fit_raw.columns
    ]

    percent_cols_present = [
        c for c in PERCENT_COLS
        if c in X_fit_raw.columns
    ]

    force_numeric = [
        c for c in FORCE_NUMERIC
        if c in X_fit_raw.columns
    ]

    force_categorical = [
        c for c in FORCE_CATEGORICAL
        if c in X_fit_raw.columns
    ]

    object_cols = (
        X_fit_raw
        .select_dtypes(include=["object"])
        .columns
        .tolist()
    )

    numeric_candidates = (
        X_fit_raw.columns
        .difference(object_cols)
        .tolist()
    )

    low_card_numeric_as_cat = []

    for col in numeric_candidates:
        if (
            col in force_numeric
            or col in count_cols_present
            or col in percent_cols_present
        ):
            continue

        nunique = X_fit_raw[col].nunique(
            dropna=True
        )

        if (
            nunique <= 30
            and nunique / len(X_fit_raw) < 0.10
        ):
            low_card_numeric_as_cat.append(col)

    categorical_cols = sorted(
        set(
            object_cols
            + low_card_numeric_as_cat
            + force_categorical
        )
    )

    categorical_cols = [
        c for c in categorical_cols
        if c not in force_numeric
    ]

    count_cols_use = [
        c for c in count_cols_present
        if c in X_fit_raw.columns
        and c not in categorical_cols
    ]

    percent_cols_use = [
        c for c in percent_cols_present
        if c in X_fit_raw.columns
        and c not in categorical_cols
    ]

    other_numeric_cols = [
        c for c in X_fit_raw.columns
        if c not in categorical_cols
        and c not in count_cols_use
        and c not in percent_cols_use
    ]

    counts_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "clip",
            ClipTransformer(upper_q=0.99)
        ),
        (
            "log1p",
            FunctionTransformer(
                safe_log1p,
                feature_names_out="one-to-one"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    percent_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "to_proportion",
            FunctionTransformer(
                to_proportion,
                feature_names_out="one-to-one"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    numeric_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_transformer = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "astype_str",
            FunctionTransformer(
                astype_str,
                feature_names_out="one-to-one"
            )
        ),
        (
            "onehot",
            make_onehot()
        )
    ])

    preprocess = ColumnTransformer(
        transformers=[
            (
                "counts",
                counts_transformer,
                count_cols_use
            ),
            (
                "percent",
                percent_transformer,
                percent_cols_use
            ),
            (
                "numeric",
                numeric_transformer,
                other_numeric_cols
            ),
            (
                "categorical",
                categorical_transformer,
                categorical_cols
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    metadata = {
        "count_cols": count_cols_use,
        "percent_cols": percent_cols_use,
        "numeric_cols": other_numeric_cols,
        "categorical_cols": categorical_cols
    }

    return preprocess, metadata


## 7) Definición exacta de V1.0 y V2.0


In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="TFG"
)
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(
        self,
        gamma=2.0,
        alpha=0.25,
        name="binary_focal_loss"
    ):
        super().__init__(name=name)
        self.gamma = float(gamma)
        self.alpha = float(alpha)

    def call(self, y_true, y_pred):
        y_true = tf.cast(
            y_true,
            tf.float32
        )

        y_true = tf.reshape(
            y_true,
            tf.shape(y_pred)
        )

        eps = tf.keras.backend.epsilon()

        y_pred = tf.clip_by_value(
            y_pred,
            eps,
            1.0 - eps
        )

        bce = tf.keras.backend.binary_crossentropy(
            y_true,
            y_pred
        )

        p_t = (
            y_true * y_pred
            + (1.0 - y_true)
            * (1.0 - y_pred)
        )

        alpha_factor = (
            y_true * self.alpha
            + (1.0 - y_true)
            * (1.0 - self.alpha)
        )

        modulating_factor = tf.pow(
            1.0 - p_t,
            self.gamma
        )

        loss = (
            alpha_factor
            * modulating_factor
            * bce
        )

        return tf.reduce_mean(
            loss,
            axis=-1
        )

    def get_config(self):
        config = super().get_config()

        config.update({
            "gamma": self.gamma,
            "alpha": self.alpha
        })

        return config


def build_v1(input_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(input_dim,)
        ),

        tf.keras.layers.Dense(
            256,
            kernel_regularizer=tf.keras.regularizers.l2(
                1e-4
            )
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(
            128,
            kernel_regularizer=tf.keras.regularizers.l2(
                1e-4
            )
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.30),

        tf.keras.layers.Dense(
            64,
            kernel_regularizer=tf.keras.regularizers.l2(
                1e-4
            )
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC"
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            )
        ]
    )

    return model


def build_v2(input_dim, alpha):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(input_dim,)
        ),

        tf.keras.layers.Dense(
            256,
            kernel_regularizer=tf.keras.regularizers.l2(
                1e-4
            )
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Dense(
            128,
            kernel_regularizer=tf.keras.regularizers.l2(
                1e-4
            )
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Dense(64),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.10),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss=BinaryFocalLoss(
            gamma=2.0,
            alpha=float(alpha)
        ),
        metrics=[
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC"
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            )
        ]
    )

    return model


## 8) Callbacks exactos de cada versión


In [ ]:
def callbacks_v1():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=20,
            restore_best_weights=True,
            verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            factor=0.5,
            patience=8,
            min_lr=1e-6,
            verbose=0
        )
    ]


def callbacks_v2():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=12,
            restore_best_weights=True,
            verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_pr_auc",
            mode="max",
            factor=0.5,
            patience=6,
            min_lr=1e-6,
            verbose=0
        )
    ]


## 9) Función de evaluación

La métrica `PR_AUC` se calcula con integración trapezoidal de la curva Precision-Recall.

`Average_Precision` se reporta por separado porque **no es exactamente la misma métrica**.


In [ ]:
def evaluate_probabilities(
    y_true,
    y_prob,
    threshold=0.50
):
    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_true,
            y_prob
        )
    )

    pr_auc = auc(
        recall_curve,
        precision_curve
    )

    average_precision = (
        average_precision_score(
            y_true,
            y_prob
        )
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    return {
        "PR_AUC": float(pr_auc),
        "Average_Precision": float(
            average_precision
        ),
        "ROC_AUC": float(roc_auc),
        "Accuracy_050": float(
            accuracy_score(
                y_true,
                y_pred
            )
        ),
        "Precision_050": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            )
        ),
        "Recall_050": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            )
        ),
        "F1_050": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        ),
        "Specificity_050": float(
            specificity
        ),
        "FNR_050": float(fnr),
        "Balanced_Accuracy_050": float(
            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),
        "MCC_050": float(
            matthews_corrcoef(
                y_true,
                y_pred
            )
        ),
        "TN_050": int(tn),
        "FP_050": int(fp),
        "FN_050": int(fn),
        "TP_050": int(tp)
    }


# 10) Ejecución de 5 folds

## Diseño dentro de cada fold

Cada fold externo reserva aproximadamente 20 % del conjunto de desarrollo como **evaluación externa del fold**.

La porción restante se divide nuevamente para obtener una **validación interna de EarlyStopping**.

Por tanto, el fold de evaluación no participa en:

- `preprocess.fit()`;
- cálculo de `class_weight`;
- cálculo de `alpha`;
- EarlyStopping;
- ReduceLROnPlateau.

Esto hace la estimación más limpia que utilizar el mismo fold simultáneamente para detener el entrenamiento y medir el rendimiento.


In [ ]:

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

results = []
fold_partitions = []

for fold, (
    outer_train_pos,
    outer_eval_pos
) in enumerate(
    skf.split(
        X_dev_raw,
        y_dev
    ),
    start=1
):
    print("\n" + "=" * 88)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 88)

    fold_seed = (
        SEED * 100
        + fold
    )

    # ---------------------------------------
    # Validación interna para EarlyStopping
    # ---------------------------------------
    (
        fit_pos,
        earlystop_pos
    ) = train_test_split(
        outer_train_pos,
        test_size=INNER_VAL_SIZE,
        random_state=fold_seed,
        stratify=y_dev[outer_train_pos]
    )

    X_fit_raw = X_dev_raw.iloc[
        fit_pos
    ].copy()

    X_earlystop_raw = X_dev_raw.iloc[
        earlystop_pos
    ].copy()

    X_eval_raw = X_dev_raw.iloc[
        outer_eval_pos
    ].copy()

    y_fit = y_dev[fit_pos]
    y_earlystop = y_dev[earlystop_pos]
    y_eval = y_dev[outer_eval_pos]

    print(
        "Tamaños -> "
        f"fit={len(fit_pos)}, "
        f"earlystop={len(earlystop_pos)}, "
        f"eval={len(outer_eval_pos)}"
    )

    # ---------------------------------------
    # PREPROCESAMIENTO: FIT SOLO EN fit
    # ---------------------------------------
    preprocess, prep_meta = (
        build_preprocessor(
            X_fit_raw
        )
    )

    preprocess.fit(
        X_fit_raw
    )

    X_fit = preprocess.transform(
        X_fit_raw
    ).astype(np.float32)

    X_earlystop = preprocess.transform(
        X_earlystop_raw
    ).astype(np.float32)

    X_eval = preprocess.transform(
        X_eval_raw
    ).astype(np.float32)

    for name, X_arr in [
        ("fit", X_fit),
        ("earlystop", X_earlystop),
        ("eval", X_eval)
    ]:
        if np.isnan(X_arr).any():
            raise ValueError(
                f"Fold {fold}: NaN en {name}"
            )

        if not np.isfinite(
            X_arr
        ).all():
            raise ValueError(
                f"Fold {fold}: infinitos en {name}"
            )

    if not (
        X_fit.shape[1]
        == X_earlystop.shape[1]
        == X_eval.shape[1]
    ):
        raise RuntimeError(
            f"Fold {fold}: dimensiones "
            "transformadas incompatibles."
        )

    fold_partitions.append({
        "Fold": fold,
        "n_fit": len(fit_pos),
        "n_earlystop": len(
            earlystop_pos
        ),
        "n_eval": len(
            outer_eval_pos
        ),
        "positive_rate_fit": float(
            np.mean(y_fit)
        ),
        "positive_rate_earlystop": float(
            np.mean(y_earlystop)
        ),
        "positive_rate_eval": float(
            np.mean(y_eval)
        ),
        "n_features_after_preprocess": int(
            X_fit.shape[1]
        )
    })

    # =======================================
    # V1.0
    # =======================================
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(
        fold_seed
    )

    classes = np.array([0, 1])

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_fit
    )

    class_weight_v1 = {
        0: float(weights[0]),
        1: float(weights[1])
    }

    model_v1 = build_v1(
        X_fit.shape[1]
    )

    history_v1 = model_v1.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_earlystop,
            y_earlystop
        ),
        epochs=MAX_EPOCHS,
        batch_size=32,
        class_weight=class_weight_v1,
        callbacks=callbacks_v1(),
        verbose=0
    )

    prob_v1 = model_v1.predict(
        X_eval,
        verbose=0
    ).ravel()

    metrics_v1 = evaluate_probabilities(
        y_eval,
        prob_v1,
        threshold=THRESHOLD_DESCRIPTIVE
    )

    results.append({
        "Fold": fold,
        "Modelo": "V1.0",
        "Seed_fold": fold_seed,
        "n_fit": len(fit_pos),
        "n_earlystop": len(
            earlystop_pos
        ),
        "n_eval": len(
            outer_eval_pos
        ),
        "n_features": int(
            X_fit.shape[1]
        ),
        "Epochs": len(
            history_v1.history["loss"]
        ),
        "Best_val_PR_AUC": float(
            np.max(
                history_v1.history[
                    "val_pr_auc"
                ]
            )
        ),
        **metrics_v1
    })

    del model_v1
    gc.collect()

    # =======================================
    # V2.0
    # =======================================
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(
        fold_seed
    )

    alpha_v2 = float(
        1.0 - np.mean(y_fit)
    )

    model_v2 = build_v2(
        X_fit.shape[1],
        alpha=alpha_v2
    )

    history_v2 = model_v2.fit(
        X_fit,
        y_fit,
        validation_data=(
            X_earlystop,
            y_earlystop
        ),
        epochs=MAX_EPOCHS,
        batch_size=64,
        callbacks=callbacks_v2(),
        verbose=0
    )

    prob_v2 = model_v2.predict(
        X_eval,
        verbose=0
    ).ravel()

    metrics_v2 = evaluate_probabilities(
        y_eval,
        prob_v2,
        threshold=THRESHOLD_DESCRIPTIVE
    )

    results.append({
        "Fold": fold,
        "Modelo": "V2.0",
        "Seed_fold": fold_seed,
        "n_fit": len(fit_pos),
        "n_earlystop": len(
            earlystop_pos
        ),
        "n_eval": len(
            outer_eval_pos
        ),
        "n_features": int(
            X_fit.shape[1]
        ),
        "Epochs": len(
            history_v2.history["loss"]
        ),
        "Best_val_PR_AUC": float(
            np.max(
                history_v2.history[
                    "val_pr_auc"
                ]
            )
        ),
        **metrics_v2
    })

    del model_v2
    gc.collect()

    print(
        f"V1 PR-AUC={metrics_v1['PR_AUC']:.4f} | "
        f"V2 PR-AUC={metrics_v2['PR_AUC']:.4f}"
    )

    print(
        f"V1 Recall@0.50={metrics_v1['Recall_050']:.4f} | "
        f"V2 Recall@0.50={metrics_v2['Recall_050']:.4f}"
    )

results_df = pd.DataFrame(
    results
)

fold_partitions_df = pd.DataFrame(
    fold_partitions
)

print("\nCross-validation completada.")
display(results_df)


## 11) Verificaciones posteriores


In [ ]:
if len(results_df) != N_SPLITS * 2:
    raise RuntimeError(
        "Se esperaban exactamente "
        f"{N_SPLITS * 2} filas de resultados."
    )

counts_by_model = (
    results_df["Modelo"]
    .value_counts()
    .to_dict()
)

if counts_by_model != {
    "V1.0": N_SPLITS,
    "V2.0": N_SPLITS
}:
    raise RuntimeError(
        f"Cantidad inesperada por modelo: "
        f"{counts_by_model}"
    )

# TEST continúa intacto.
assert len(y_test_frozen) == 150
assert X_test_frozen_raw.shape[0] == 150

print("Validaciones posteriores OK.")
print(
    "TEST congelado no fue transformado "
    "ni evaluado por ningún modelo en el CV."
)


# 12) Resumen media ± desviación estándar

La tabla siguiente es la salida principal que debe revisarse con el tutor.

**PR-AUC** es la métrica principal. Recall, MCC y Balanced Accuracy complementan la interpretación.


In [ ]:
metrics_to_summarize = [
    "PR_AUC",
    "Average_Precision",
    "ROC_AUC",
    "Accuracy_050",
    "Precision_050",
    "Recall_050",
    "F1_050",
    "Specificity_050",
    "FNR_050",
    "Balanced_Accuracy_050",
    "MCC_050",
    "FN_050",
    "FP_050",
    "Epochs",
    "Best_val_PR_AUC"
]

summary_rows = []

for model_name in [
    "V1.0",
    "V2.0"
]:
    sub = results_df[
        results_df["Modelo"]
        == model_name
    ]

    for metric in metrics_to_summarize:
        summary_rows.append({
            "Modelo": model_name,
            "Métrica": metric,
            "Media": float(
                sub[metric].mean()
            ),
            "Desv_Estandar": float(
                sub[metric].std(
                    ddof=1
                )
            ),
            "Min": float(
                sub[metric].min()
            ),
            "Max": float(
                sub[metric].max()
            )
        })

summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df.style.format({
        "Media": "{:.4f}",
        "Desv_Estandar": "{:.4f}",
        "Min": "{:.4f}",
        "Max": "{:.4f}"
    })
)


## 13) Comparación pareada fold por fold


In [ ]:
paired_metrics = [
    "PR_AUC",
    "Average_Precision",
    "ROC_AUC",
    "Recall_050",
    "F1_050",
    "Balanced_Accuracy_050",
    "MCC_050",
    "FNR_050",
    "FN_050"
]

paired_rows = []

for metric in paired_metrics:
    pivot = results_df.pivot(
        index="Fold",
        columns="Modelo",
        values=metric
    )

    pivot["Delta_V2_menos_V1"] = (
        pivot["V2.0"]
        - pivot["V1.0"]
    )

    for fold, row in pivot.iterrows():
        paired_rows.append({
            "Métrica": metric,
            "Fold": int(fold),
            "V1.0": float(
                row["V1.0"]
            ),
            "V2.0": float(
                row["V2.0"]
            ),
            "Delta_V2_menos_V1": float(
                row[
                    "Delta_V2_menos_V1"
                ]
            )
        })

paired_df = pd.DataFrame(
    paired_rows
)

display(paired_df)


## 14) Tabla compacta para discusión con el tutor

Esta tabla coloca V1 y V2 lado a lado como **media ± desviación estándar**.


In [ ]:
key_metrics = [
    "PR_AUC",
    "Average_Precision",
    "ROC_AUC",
    "Recall_050",
    "F1_050",
    "Balanced_Accuracy_050",
    "MCC_050",
    "FNR_050"
]

compact_rows = []

for metric in key_metrics:
    row = {
        "Métrica": metric
    }

    for model_name in [
        "V1.0",
        "V2.0"
    ]:
        sub = results_df[
            results_df["Modelo"]
            == model_name
        ][metric]

        row[model_name] = (
            f"{sub.mean():.4f} "
            f"± {sub.std(ddof=1):.4f}"
        )

    compact_rows.append(row)

compact_df = pd.DataFrame(
    compact_rows
)

display(compact_df)


## 15) Gráficos de estabilidad


In [ ]:
def metric_mean_std(metric):
    stats = (
        results_df
        .groupby("Modelo")[metric]
        .agg(["mean", "std"])
        .reindex(["V1.0", "V2.0"])
    )
    return stats


# PR-AUC
stats = metric_mean_std(
    "PR_AUC"
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.bar(
    stats.index,
    stats["mean"],
    yerr=stats["std"],
    capsize=6
)

ax.set_ylabel("PR-AUC")
ax.set_title(
    "CV-5 — PR-AUC media ± desviación estándar"
)
ax.set_ylim(0, 1.02)
ax.grid(
    axis="y",
    alpha=0.2
)
plt.tight_layout()
plt.show()


In [ ]:
# Recall a threshold 0.50 — métrica secundaria
stats = metric_mean_std(
    "Recall_050"
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.bar(
    stats.index,
    stats["mean"],
    yerr=stats["std"],
    capsize=6
)

ax.set_ylabel("Recall")
ax.set_title(
    "CV-5 — Recall@0.50 media ± desviación estándar"
)
ax.set_ylim(0, 1.02)
ax.grid(
    axis="y",
    alpha=0.2
)
plt.tight_layout()
plt.show()


In [ ]:
# MCC a threshold 0.50 — métrica secundaria
stats = metric_mean_std(
    "MCC_050"
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

ax.bar(
    stats.index,
    stats["mean"],
    yerr=stats["std"],
    capsize=6
)

ax.set_ylabel("MCC")
ax.set_title(
    "CV-5 — MCC@0.50 media ± desviación estándar"
)
ax.grid(
    axis="y",
    alpha=0.2
)
plt.tight_layout()
plt.show()


## 16) Exportación

Los resultados se guardan en la carpeta de salida configurada para incorporarlos posteriormente al registro maestro y al documento del TFG.


In [ ]:
results_df.to_csv(
    OUTPUT_DIR / "resultados_cv5_por_fold.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    OUTPUT_DIR / "resumen_cv5_media_sd.csv",
    index=False,
    encoding="utf-8-sig"
)

paired_df.to_csv(
    OUTPUT_DIR / "comparacion_pareada_cv5.csv",
    index=False,
    encoding="utf-8-sig"
)

compact_df.to_csv(
    OUTPUT_DIR / "tabla_compacta_cv5.csv",
    index=False,
    encoding="utf-8-sig"
)

fold_partitions_df.to_csv(
    OUTPUT_DIR / "particiones_cv5.csv",
    index=False,
    encoding="utf-8-sig"
)

snapshot = {
    "experiment_id": "EXP-005",
    "fecha_ejecucion": (
        datetime.now().isoformat()
    ),
    "dataset_path": DATASET_PATH,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "inner_validation_size": (
        INNER_VAL_SIZE
    ),
    "development_n": int(
        len(y_dev)
    ),
    "frozen_test_n": int(
        len(y_test_frozen)
    ),
    "frozen_test_used_for_training": False,
    "frozen_test_used_for_evaluation": False,
    "primary_metric": "PR_AUC",
    "secondary_threshold": (
        THRESHOLD_DESCRIPTIVE
    ),
    "methodological_note": (
        "5-fold externo sobre TRAIN+VALIDATION originales. "
        "Cada outer-train se divide internamente para EarlyStopping. "
        "El preprocesamiento se ajusta únicamente con el fit interno. "
        "TEST original permanece congelado."
    )
}

with open(
    OUTPUT_DIR / "snapshot_cv5.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        snapshot,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Archivos guardados en:")
print(OUTPUT_DIR)


# 17) Cómo interpretar el resultado

No se debe elegir automáticamente una versión por una diferencia mínima.

La revisión debe seguir este orden:

1. Comparar **PR-AUC media**.
2. Revisar su **desviación estándar**.
3. Revisar ROC-AUC y Average Precision.
4. Como métricas operativas secundarias, revisar Recall, FNR, MCC y Balanced Accuracy a threshold 0.50.
5. Revisar si la ventaja de un modelo aparece de manera consistente en varios folds o depende de uno solo.

## Después de ejecutar

Comparte:

- `resultados_cv5_por_fold.csv`
- `resumen_cv5_media_sd.csv`
- `tabla_compacta_cv5.csv`

Con esos tres archivos se actualizará el `Registro_Maestro_Experimentos_MLP_TFG.xlsx` y se documentará la decisión correspondiente.

> **No ejecutar todavía una nueva evaluación sobre TEST para decidir entre V1 y V2.**
